# CAD font matching lab

Test harness for step 5 (candidate graph construction) and step 6 (affine subgraph matching + step 6.5 scoring) of `docs/cad_font_vector_recognition.md`, scoped to **one hand-picked character template vs one hand-picked test candidate cluster** -- not the full multi-template/multi-candidate pipeline (steps 4, 6.1-6.2, 7-12 are out of scope; those need real clustering and cross-candidate baseline-confidence state this harness doesn't build).

A character bank is built the same way `cad_font_char_graph_lab.ipynb` does (from a `scripts/label/CAD_font_label.py` output file). The test candidate is a **single labelled cluster from a `scripts/label/vector_label.py` output file** -- a controlled stand-in for step 4's real seqno-spatial clustering, so the matching math (transform fitting, correspondence search, scoring) can be visually validated in isolation before steps 4/7-12 are attempted.

## 0 - Config

In [ ]:
from pathlib import Path

CAD_FONT_LABEL_JSON_PATH = None   # required: CAD_font_label.py output json (character bank source)
                                   # (default outputs/labels/<stem>_cad_font.json)
TEST_LABEL_JSON_PATH = None       # required: vector_label.py output json -- a single labelled
                                   # candidate cluster to test matching against
TEST_LABEL_ID = None              # None -> assume the test file has exactly one source="vector" entry

CHARACTER_LABEL_ID = None         # pick the character template by exact CharacterTemplate.label_id
CHARACTER_TEXT = None             # OR by text (first matching template wins if several share it)
RANDOM_SEED = None                # OR: neither of the above set -> random pick, seeded for reproducibility

POINT_MERGE_TOL = 1e-3          # coincident-point dedup tolerance (pdf pts)
CURVE_SPACING_FRACTION = 0.03   # adaptive curve-sampling spacing, as a fraction of each shape's own bbox height
AREA_TOL_FRACTION = 0.0005      # simplification area-drop threshold, as a fraction of (bbox height)**2

OVERLAY_DPI = 300        # render_vector_cluster raster dpi for visualization

# step 6.5 scoring weights (matching.score_match) -- tune here, not in matching.py
MSE_WEIGHT = 1.0
EDGE_WEIGHT = 1.0
EDGE_DEGREE_BLEND = 0.5

TOP_K_MATCHES = 100      # render/save only the TOP_K_MATCHES best matches by score.total (lower = better)

OUTPUT_DIR = None        # None -> alongside TEST_LABEL_JSON_PATH, "<stem>_matches"


## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)


## 2 - Load the character bank

Same character-bank construction as `cad_font_char_graph_lab.ipynb`: a `CAD_font_label.py` output JSON, re-extracting vectors per page from its own `pdf_path`.

In [ ]:
from rastervec.Evaluation.Labelling.label_schema import load_labels
from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.vector_extract import extract_vectors
from rastervec.Evaluation.CadFont.character_bank import build_character_bank

assert CAD_FONT_LABEL_JSON_PATH, "set CAD_FONT_LABEL_JSON_PATH"
bank_labels = load_labels(CAD_FONT_LABEL_JSON_PATH)
assert bank_labels.pdf_path, f"{CAD_FONT_LABEL_JSON_PATH} has no pdf_path recorded"

cad_entries = [e for e in bank_labels.entries if e.source == "cad_font"]
print(f"{len(cad_entries)} cad_font character label(s), {len(bank_labels.baselines)} baseline(s)")

bank_reader = Reader(bank_labels.pdf_path)
bank_pages_needed = sorted({e.page_index for e in cad_entries})
bank_vectors_by_page = {p: extract_vectors(bank_reader.get_page(p)) for p in bank_pages_needed}
bank_entries_by_label_id = {e.label_id: e for e in cad_entries}
bank_baselines_by_id = {b.baseline_id: b for b in bank_labels.baselines}

templates = build_character_bank(
    bank_labels, bank_vectors_by_page,
    point_merge_tol=POINT_MERGE_TOL,
    curve_spacing_fraction=CURVE_SPACING_FRACTION,
    area_tol_fraction=AREA_TOL_FRACTION,
)
print(f"{len(templates)} character template(s) built (of {len(cad_entries)} cad_font entries)")


## 3 - Load the test candidate cluster

A `scripts/label/vector_label.py` output JSON, assumed to hold exactly one relevant `source="vector"` entry -- the single hand-picked candidate cluster to test matching against (a controlled stand-in for step 4's real clustering). Opens its own `Reader`, since the test PDF may differ from the character-bank PDF.

In [ ]:
from rastervec.Evaluation.Labelling.label_schema import path_signature

assert TEST_LABEL_JSON_PATH, "set TEST_LABEL_JSON_PATH"
test_labels = load_labels(TEST_LABEL_JSON_PATH)
assert test_labels.pdf_path, f"{TEST_LABEL_JSON_PATH} has no pdf_path recorded"

test_entries = [e for e in test_labels.entries if e.source == "vector"]
if TEST_LABEL_ID is not None:
    test_entry = next(e for e in test_entries if e.label_id == TEST_LABEL_ID)
else:
    assert len(test_entries) == 1, (
        f"expected exactly one source='vector' entry in {TEST_LABEL_JSON_PATH}, "
        f"found {len(test_entries)} -- set TEST_LABEL_ID to disambiguate"
    )
    test_entry = test_entries[0]

test_reader = Reader(test_labels.pdf_path)
test_page_vectors = extract_vectors(test_reader.get_page(test_entry.page_index))
test_sig_map = {path_signature(v): v for v in test_page_vectors}
resolved_test_vectors = [test_sig_map[s] for s in test_entry.vector_signatures if s in test_sig_map]
assert resolved_test_vectors, "no vector_signatures from the test label resolved against the test PDF"
print(f"test candidate cluster: {len(resolved_test_vectors)} vector(s), page {test_entry.page_index}")


## 4 - Pick a character template

In [ ]:
import random

def _pick_template():
    if CHARACTER_LABEL_ID is not None:
        for t in templates:
            if t.label_id == CHARACTER_LABEL_ID:
                return t
        raise ValueError(f"no template with label_id={CHARACTER_LABEL_ID!r}")
    if CHARACTER_TEXT is not None:
        for t in templates:
            if t.text == CHARACTER_TEXT:
                return t
        raise ValueError(f"no template with text={CHARACTER_TEXT!r}")
    rng = random.Random(RANDOM_SEED)
    return rng.choice(templates)

chosen = _pick_template()
print(
    f'chosen character: "{chosen.text}"  label_id={chosen.label_id}  '
    f'complexity={chosen.complexity:.0f}  nodes={chosen.graph.num_nodes()}  '
    f'edges={chosen.graph.num_edges()}  anchor_node_indices={chosen.anchor_node_indices}'
)


## 5 - Critical points

"Critical points" = `critical_point_indices(chosen.graph) | chosen.graph.synthetic_anchor_indices` -- the chosen template's own eligible/anchor-search pool (`original_vertex_indices | {degree > 2 nodes}`, plus any synthesized 3rd anchor), split into 3 colors here: **red** = endpoint (degree == 1), **orange** = junction (degree > 2), **magenta** = synthetic anchor.

Step 6.5's scoring (section 7 below) is computed specifically over these points -- matched into the candidate graph's *full* node set, not just its own critical points (a template critical point can legitimately correspond to a non-critical candidate node).

In [ ]:
import math
import matplotlib.pyplot as plt

from rastervec.Evaluation.CadFont.character_bank import to_baseline_relative_vectors
from rastervec.Evaluation.CadFont.graph import critical_point_indices
from rastervec.commons.helpers.geometry import item_points, transform_point, union_bbox
from rastervec.commons.renderer.png import render_vector_cluster, page_points_to_pixel

chosen_entry = bank_entries_by_label_id[chosen.label_id]
chosen_baseline = bank_baselines_by_id[chosen.baseline_id]
chosen_sig_map = {path_signature(v): v for v in bank_vectors_by_page[chosen_entry.page_index]}
chosen_resolved = [chosen_sig_map[s] for s in chosen_entry.vector_signatures if s in chosen_sig_map]

# Solve the inverse of to_baseline_relative_vectors's rigid transform the same
# way cad_font_char_graph_lab.ipynb does: one known (page, baseline-relative)
# point pair is enough, rather than re-deriving the forward transform's own
# internal min_x shift by hand.
chosen_tvecs = to_baseline_relative_vectors(chosen_resolved, chosen_baseline.origin, chosen_baseline.direction)
chosen_theta = math.degrees(math.atan2(chosen_baseline.direction[1], chosen_baseline.direction[0]))
_p_page0 = item_points(chosen_resolved[0].items[0])[0]
_p_rel0 = item_points(chosen_tvecs[0].items[0])[0]
_r_rel0 = transform_point(_p_rel0, offset=(0.0, 0.0), rotation_deg=chosen_theta)
chosen_t_inv = (_p_page0[0] - _r_rel0[0], _p_page0[1] - _r_rel0[1])

chosen_graph = chosen.graph
critical = set(critical_point_indices(chosen_graph)) | set(chosen_graph.synthetic_anchor_indices)
_degrees = chosen_graph.degrees()

def _point_category(i):
    if i in chosen_graph.synthetic_anchor_indices:
        return "synthetic"
    if _degrees[i] > 2:
        return "junction"
    return "endpoint"

_category_color = {"endpoint": "red", "junction": "orange", "synthetic": "magenta"}

page_pts = [transform_point(n, offset=chosen_t_inv, rotation_deg=chosen_theta) for n in chosen_graph.nodes]

# render_vector_cluster/page_points_to_pixel size their frame from the
# VECTORS' own bbox + a flat padding -- a synthesized anchor (or any other
# graph node) can legitimately land outside that bbox, so grow the padding
# by however far the graph's own nodes overhang it on any side, guaranteeing
# every node -- not just the vector geometry -- lands inside the rendered
# window. Both calls below must keep sharing this same padding value.
_vec_bbox = union_bbox([v.bbox for v in chosen_resolved])
_node_xs = [p[0] for p in page_pts]
_node_ys = [p[1] for p in page_pts]
_overhang = max(
    _vec_bbox[0] - min(_node_xs), _vec_bbox[1] - min(_node_ys),
    max(_node_xs) - _vec_bbox[2], max(_node_ys) - _vec_bbox[3],
    0.0,
)
render_padding = 2.0 + _overhang

img = render_vector_cluster(chosen_resolved, dpi=OVERLAY_DPI, padding=render_padding)
pixel_pts = page_points_to_pixel(chosen_resolved, OVERLAY_DPI, page_pts, padding=render_padding)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)
for a, b in chosen_graph.edges:
    (x0, y0), (x1, y1) = pixel_pts[a], pixel_pts[b]
    ax.plot([x0, x1], [y0, y1], color="lime", linewidth=1.0, zorder=2)
for i, (x, y) in enumerate(pixel_pts):
    if i in critical:
        ax.scatter([x], [y], c=_category_color[_point_category(i)], s=40, zorder=3, edgecolors="black", linewidths=0.5)
    else:
        ax.scatter([x], [y], c="lightgrey", s=15, zorder=2.5, edgecolors="grey", linewidths=0.3)
ax.set_title(f'"{chosen.text}" critical points -- red=endpoint orange=junction magenta=synthetic  (grey=non-critical)')
ax.axis("off")
plt.show()


## 6 - Candidate graph (step 5, page space)

`build_candidate_graph` runs the same flatten/split/merge/simplify pipeline a template gets, but skips `to_baseline_relative_vectors` -- the result stays in raw page space, since no baseline is known yet for an unlabelled candidate.

In [ ]:
from rastervec.Evaluation.CadFont.matching import build_candidate_graph

candidate_graph = build_candidate_graph(
    resolved_test_vectors,
    point_merge_tol=POINT_MERGE_TOL,
    curve_spacing_fraction=CURVE_SPACING_FRACTION,
    area_tol_fraction=AREA_TOL_FRACTION,
)
candidate_critical = critical_point_indices(candidate_graph)
print(
    f"candidate graph: nodes={candidate_graph.num_nodes()} edges={candidate_graph.num_edges()} "
    f"max_degree={candidate_graph.max_degree()} critical_points={len(candidate_critical)}"
)

cand_img = render_vector_cluster(resolved_test_vectors, dpi=OVERLAY_DPI, padding=2.0)
cand_pixel_pts = page_points_to_pixel(resolved_test_vectors, OVERLAY_DPI, candidate_graph.nodes, padding=2.0)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(cand_img)
for a, b in candidate_graph.edges:
    (x0, y0), (x1, y1) = cand_pixel_pts[a], cand_pixel_pts[b]
    ax.plot([x0, x1], [y0, y1], color="blue", linewidth=1.0, zorder=2)
xs, ys = zip(*cand_pixel_pts)
ax.scatter(xs, ys, c="red", s=25, zorder=3, edgecolors="black", linewidths=0.5)
ax.set_title(f"candidate graph (page space) -- {len(resolved_test_vectors)} vector(s)")
ax.axis("off")
plt.show()


## 7 - Step 6 matching

`match_template_against_candidate` (step 6.3-6.5): enumerates every ordered, injective assignment of the chosen template's 3 anchors to the candidate graph's own critical points, filtered per-slot by "candidate degree >= that anchor's own degree" (`enumerate_anchor_correspondences`); fits a constrained similarity transform (uniform scale + rotation + translation -- no shear possible by construction, satisfying step 6.3.2) from each correspondence via a closed-form Umeyama least-squares fit; maps every template node into candidate space and nearest-neighbor-matches it against the candidate graph's *entire* node set; scores each result (step 6.5) specifically over the template's own critical points.

This cell tests **every** correspondence found -- no similarity threshold -- then sorts them by `score.total` (lower = better) and keeps only the top `TOP_K_MATCHES`. The next cell renders them in that same best-first order.

In [ ]:
from rastervec.Evaluation.CadFont.matching import match_template_against_candidate

matches = match_template_against_candidate(
    chosen, candidate_graph,
    mse_weight=MSE_WEIGHT, edge_weight=EDGE_WEIGHT, edge_degree_blend=EDGE_DEGREE_BLEND,
)
print(f'{len(matches)} correspondence(s) tested for "{chosen.text}" vs the candidate graph')

matches = sorted(matches, key=lambda m: m.score.total)[:TOP_K_MATCHES]
print(f'keeping top {len(matches)} match(es) by score.total (lower = better)')


## 8 - Render top matches

Renders the top `TOP_K_MATCHES` matches from the previous cell, best (lowest `score.total`) first. Legend per rendered result: **grey** = every unmatched candidate node/edge; **red/blue** = the matched candidate subgraph's nodes/edges; **green** = the *entire* transformed template graph overlaid (not just its critical points); **dashed green** = the template's own baseline (`y=0` in its baseline-relative frame), mapped through the fitted transform and clipped to the rendered cluster's own padded bbox. Title shows the rank, correspondence, and full score breakdown (lower `total`/`mse`/`edge` = better).

In [ ]:
from rastervec.commons.helpers.geometry import clip_line_to_bbox, union_bbox

if OUTPUT_DIR:
    OUTPUT_DIR_PATH = Path(OUTPUT_DIR)
else:
    OUTPUT_DIR_PATH = Path(TEST_LABEL_JSON_PATH).parent / (Path(TEST_LABEL_JSON_PATH).stem + "_matches")
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

_cand_bbox = union_bbox([v.bbox for v in resolved_test_vectors])
_x0, _y0, _x1, _y1 = _cand_bbox
_padded_bbox = (_x0 - 2.0, _y0 - 2.0, _x1 + 2.0, _y1 + 2.0)

for idx, match in enumerate(matches):
    matched_candidate_nodes = set(match.node_map.values())
    matched_candidate_edges = {
        (a, b) for a, b in candidate_graph.edges
        if a in matched_candidate_nodes and b in matched_candidate_nodes
    }

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(cand_img)

    for a, b in candidate_graph.edges:
        if (a, b) in matched_candidate_edges:
            continue
        (x0, y0), (x1, y1) = cand_pixel_pts[a], cand_pixel_pts[b]
        ax.plot([x0, x1], [y0, y1], color="grey", linewidth=0.8, zorder=1)
    for i, (x, y) in enumerate(cand_pixel_pts):
        if i in matched_candidate_nodes:
            continue
        ax.scatter([x], [y], c="grey", s=15, zorder=1)

    for a, b in matched_candidate_edges:
        (x0, y0), (x1, y1) = cand_pixel_pts[a], cand_pixel_pts[b]
        ax.plot([x0, x1], [y0, y1], color="blue", linewidth=1.5, zorder=2)
    for i in matched_candidate_nodes:
        x, y = cand_pixel_pts[i]
        ax.scatter([x], [y], c="red", s=30, zorder=3, edgecolors="black", linewidths=0.5)

    transformed_page_pts = match.transform.apply_many(chosen_graph.nodes)
    transformed_pixel_pts = page_points_to_pixel(resolved_test_vectors, OVERLAY_DPI, transformed_page_pts, padding=2.0)
    for a, b in chosen_graph.edges:
        (x0, y0), (x1, y1) = transformed_pixel_pts[a], transformed_pixel_pts[b]
        ax.plot([x0, x1], [y0, y1], color="green", linewidth=1.5, zorder=4)
    gxs, gys = zip(*transformed_pixel_pts)
    ax.scatter(gxs, gys, c="green", s=25, zorder=5, edgecolors="black", linewidths=0.5)

    origin_page = match.transform.apply((0.0, 0.0))
    dir_probe = match.transform.apply((1.0, 0.0))
    dx, dy = dir_probe[0] - origin_page[0], dir_probe[1] - origin_page[1]
    length = math.hypot(dx, dy) or 1.0
    direction_page = (dx / length, dy / length)
    clipped = clip_line_to_bbox(origin_page, direction_page, _padded_bbox)
    if clipped is not None:
        baseline_pixel_pts = page_points_to_pixel(resolved_test_vectors, OVERLAY_DPI, list(clipped), padding=2.0)
        (bx0, by0), (bx1, by1) = baseline_pixel_pts
        ax.plot([bx0, bx1], [by0, by1], color="green", linestyle="--", linewidth=1.2, zorder=4)

    ax.set_title(
        f'#{idx+1}/{len(matches)}  "{chosen.text}"  correspondence={match.correspondence}  '
        f'total={match.score.total:.4f}  mse={match.score.mse_term:.4f}  '
        f'edge={match.score.edge_term:.4f}  scale={match.transform.scale:.3f}  '
        f'rot={match.transform.rotation_deg:.1f}deg'
    )
    ax.axis("off")
    fig.savefig(OUTPUT_DIR_PATH / f"match_{idx:03d}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 9 - Save debug stats

In [ ]:
import json

match_rows = [
    {
        "index": idx,
        "correspondence": match.correspondence,
        "anchor_indices": match.anchor_indices,
        "scale": match.transform.scale,
        "rotation_deg": match.transform.rotation_deg,
        "translation": match.transform.translation,
        "score_total": match.score.total,
        "score_mse_term": match.score.mse_term,
        "score_edge_presence_cost": match.score.edge_presence_cost,
        "score_degree_cost": match.score.degree_cost,
    }
    for idx, match in enumerate(matches)
]
stats_path = OUTPUT_DIR_PATH / "matches.json"
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(
        {"character_label_id": chosen.label_id, "character_text": chosen.text, "matches": match_rows},
        f, indent=2, default=str,
    )
print(f"wrote {stats_path}")
